In [ ]:
import pandas as pd
import numpy as np
import joblib
import shap
import matplotlib.pyplot as plt
import sys
sys.path.append('../')

In [2]:
bundle = joblib.load('data/models/xgboost_production.pkl')
model = bundle['model']
threshold = bundle['threshold']

X_test = pd.read_parquet('data/processed/X_test.parquet')
Y_test = pd.read_parquet('data/processed/Y_test.parquet').iloc[:,0]

print(f"Model threshold: {threshold:.4f}")
print(f"X_test shape: {X_test.shape}")

Model threshold: 0.2161
X_test shape: (118108, 445)


In [3]:
from src.explainibility.shap_explainer import (
    build_explainer, compute_shap_values,
    get_global_importance, explain_single_prediction,
    save_explainer
)

X_train = pd.read_parquet('data/processed/X_train.parquet')
background_sample = X_train.sample(500, random_state=42)
del X_train

explainer = build_explainer(model, background_sample)
save_explainer(explainer, 'data/models/shap_explainer.pkl')

X_test_sample = X_test.sample(5000, random_state= 42)
shap_values = compute_shap_values(explainer, X_test_sample)



Building TreeExplainer...
explainer built.
Explainer saved to data/models/shap_explainer.pkl
Computing shap values for 5,000 rows
SHAP values shape: (5000, 445)


In [4]:
importance = get_global_importance(
    shap_values,
    feature_names= X_test_sample.columns.tolist(),
    top_n = 20
)

print(f"\n Top 20 features by mean absolute SHAP value:")
print(importance.to_string(index=False))




 Top 20 features by mean absolute SHAP value:
           feature  importance
               C13    0.567824
              V317    0.323913
        card1_freq    0.301665
    TransactionAmt    0.300702
                M4    0.292110
               C14    0.281833
P_emaildomain_freq    0.265036
    card1_count_1h    0.255471
                C1    0.252681
             card1    0.244065
     P_emaildomain    0.217027
        card4_freq    0.216210
 card1_mean_amt_1h    0.209943
               C11    0.204166
                D1    0.204115
             addr1    0.203734
           tx_hour    0.199242
        card2_freq    0.195481
     amt_deviation    0.183423
              V258    0.181538


In [6]:
Y_prob = model.predict_proba(X_test)[:, 1]
high_risk_idx = Y_prob.argmax()

X_single = X_test.iloc[[high_risk_idx]]
fraud_prob = Y_prob[high_risk_idx]

print(f"Transaction fraud probability: {fraud_prob:.4f}")
print(f"Decision at threshold {threshold:.4f}: "
      f"{'FRAUD' if fraud_prob >= threshold else 'LEGITIMATE'}")

explanation = explain_single_prediction(explainer, X_single, top_n=10)

print(f"\nBase value (average prediction): {explanation['base_value']:.4f}")
print(f"This prediction:                 {explanation['prediction']:.4f}")
print(f"\nTop 10 features driving this decision:")
print(f"{'Feature':<35} {'SHAP value':>12} {'Actual value':>15}")
print("-" * 65)
for feat in explanation['top_features']:
    direction = "↑ fraud" if feat['shap_value'] > 0 else "↓ legit"
    print(f"{feat['features']:<35} "
          f"{feat['shap_value']:>+12.4f} "
          f"{str(feat['actual_value']):>15}  {direction}")

Transaction fraud probability: 1.0000
Decision at threshold 0.2161: FRAUD

Base value (average prediction): -2.2950
This prediction:                 14.6788

Top 10 features driving this decision:
Feature                               SHAP value    Actual value
-----------------------------------------------------------------
V258                                     +1.2217             4.0  ↑ fraud
C14                                      +1.0453             0.0  ↑ fraud
TransactionAmt                           +0.9584           300.0  ↑ fraud
V187                                     +0.7868             4.0  ↑ fraud
V256                                     +0.7837             3.0  ↑ fraud
C11                                      +0.7336             4.0  ↑ fraud
V251                                     +0.7054             3.0  ↑ fraud
V149                                     +0.5994             4.0  ↑ fraud
id_30_freq                               +0.5823          0.0006  ↑ fraud
C1    